# Presupuesto de contexto para RAG

[Português (Brasil)](orcamento_contexto.ipynb) · [English](orcamento_contexto.en.ipynb)

Selecciona fragmentos por pertinencia, limita cada fuente y comprueba el tamaño del JSON que se enviará.

[Artículo completo](../article/ARTICLE.es.md)

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

cwd = Path.cwd()
lab = next(
    path
    for path in (cwd, cwd.parent, cwd / "0013_orcamento_contexto")
    if path.name == "0013_orcamento_contexto" and (path / "src" / "main.py").is_file()
)

def run(*options):
    completed = subprocess.run(
        [sys.executable, "-X", "utf8", str(lab / "src" / "main.py"), *options],
        capture_output=True, check=True, text=True, encoding="utf-8",
    )
    return json.loads(completed.stdout)

Ejecuta la siguiente celda para reproducir el ejercicio y comprobar los resultados con aserciones.

Con 245 bytes, el campo `context` codificado ocupa 214 bytes y contiene dos fragmentos. Revisa `skipped` para distinguir la cuota de fuente del límite de bytes. Con 120 bytes solo cabe `refund-summary`.

In [ ]:
report = run("--bytes", "245", "--per-source", "1")
assert len(report["context"].encode("utf-8")) == report["bytes_used"] == 214
assert [item["id"] for item in json.loads(report["context"])["documents"]] == [
    "refund-summary", "refund-window"
]
assert {item["reason"] for item in report["skipped"]} == {"source_limit", "byte_budget"}
smaller = run("--bytes", "120")
assert [item["id"] for item in json.loads(smaller["context"])["documents"]] == ["refund-summary"]
report